In [21]:
# ===== DIRECT TEST (Bypass API) =====
print("="*80)
print("🧪 DIRECT FUNCTION TEST - Bypass HTTP Server")
print("="*80)

try:
    print("\n✅ Testing retrieve_with_fallback()...")
    test_q = "Có những sự kiện nào sắp tới?"
    hits = retrieve_with_fallback(test_q, top_k=5, role="user")
    print(f"   Retrieved {len(hits)} chunks")
    for i, h in enumerate(hits[:2], 1):
        print(f"   [{i}] Score: {h['score']:.3f}, Type: {h['meta'].get('type')}")
    
    print("\n✅ Testing call_llm_groq()...")
    print("   Generating response from LLM...")
    response_parts = []
    for token in call_llm_groq(test_q, hits, role="user"):
        response_parts.append(token)
    
    full_response = "".join(response_parts)
    print(f"   ✓ Response length: {len(full_response)} characters")
    print(f"\n📝 Response Preview:")
    print(f"   {full_response[:400]}...")
    print("\n✅ DIRECT FUNCTION TEST PASSED!")
    
except Exception as e:
    print(f"❌ Error: {type(e).__name__}: {e}")
    import traceback
    traceback.print_exc()


🧪 DIRECT FUNCTION TEST - Bypass HTTP Server

✅ Testing retrieve_with_fallback()...
❌ Error: AttributeError: 'NoneType' object has no attribute 'ntotal'


Traceback (most recent call last):
  File "C:\Users\ADMIN\AppData\Local\Temp\ipykernel_29260\586798897.py", line 9, in <module>
    hits = retrieve_with_fallback(test_q, top_k=5, role="user")
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\ADMIN\AppData\Local\Temp\ipykernel_29260\441257187.py", line 81, in retrieve_with_fallback
    hits = retrieve(question, top_k=top_k, role=role)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\ADMIN\AppData\Local\Temp\ipykernel_29260\441257187.py", line 65, in retrieve
    return _retrieve_from(kb_index, kb_chunks, question, top_k)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\ADMIN\AppData\Local\Temp\ipykernel_29260\441257187.py", line 13, in _retrieve_from
    k = min(index_obj.ntotal, max(top_k * 50, 50))
            ^^^^^^^^^^^^^^^^
AttributeError: 'NoneType' object has no attribute 'ntotal'


In [31]:
# ===== FINAL VERIFICATION: Auto-Update System Status =====
import requests
import time
from datetime import datetime

print("="*80)
print("🎯 VERIFYING RAG AUTO-UPDATE SYSTEM")
print("="*80)

try:
    # 1️⃣ Check health
    print("\n✅ [1/4] Checking API Health...")
    health_response = requests.get("http://127.0.0.1:8000/health", timeout=2)
    if health_response.status_code == 200:
        print("    ✓ API Server: ONLINE")
        print(f"    ✓ Response: {health_response.json()}")
    else:
        print("    ✗ API Server: ERROR")
except Exception as e:
    print(f"    ✗ Connection Error: {e}")

try:
    # 2️⃣ Check system status
    print("\n✅ [2/4] Getting System Status...")
    status_response = requests.get("http://127.0.0.1:8000/status", timeout=2)
    if status_response.status_code == 200:
        status_data = status_response.json()
        print(f"    ✓ System Status: READY")
        print(f"    ✓ Last Updated: {status_data.get('last_updated', 'Never')}")
        print(f"    ✓ Total Updates: {status_data.get('total_updates', 0)}")
        print(f"    ✓ Auto-Update Interval: {status_data.get('auto_update_interval')}")
        print(f"    ✓ Currently Updating: {status_data.get('is_updating')}")
except Exception as e:
    print(f"    ✗ Status Check Error: {e}")

try:
    # 3️⃣ Test a simple query
    print("\n✅ [3/4] Testing RAG Query (User Role)...")
    ask_response = requests.post(
        "http://127.0.0.1:8000/ask",
        json={
            "question": "Có những sự kiện nào sắp tới?",
            "role": "user",
            "top_k": 5
        },
        timeout=15
    )
    if ask_response.status_code == 200:
        ask_data = ask_response.json()
        answer = ask_data.get('answer', '')
        print(f"    ✓ Query: '{ask_data.get('question')}'")
        print(f"    ✓ Retrieved {ask_data.get('total_hits', 0)} relevant chunks")
        print(f"    ✓ Response Length: {len(answer)} characters")
        print(f"    ✓ Last System Update: {ask_data.get('last_updated', 'Never')}")
        if answer:
            print(f"\n📝 Sample Answer (first 300 chars):\n{answer[:300]}...")
    else:
        print(f"    ✗ Query Error: {ask_response.status_code}")
except Exception as e:
    print(f"    ✗ Query Error: {e}")

try:
    # 4️⃣ Test a manual rebuild trigger
    print("\n✅ [4/4] Testing Manual Rebuild Trigger...")
    rebuild_response = requests.post(
        "http://127.0.0.1:8000/rebuild",
        timeout=2
    )
    if rebuild_response.status_code == 200:
        rebuild_data = rebuild_response.json()
        print(f"    ✓ Rebuild Status: {rebuild_data.get('status')}")
        print(f"    ✓ Message: {rebuild_data.get('message')}")
        print(f"    ✓ Previous Update: {rebuild_data.get('previous_update', 'None')}")
except Exception as e:
    print(f"    ✗ Rebuild Error: {e}")

print("\n" + "="*80)
print("✅ VERIFICATION COMPLETE!")
print("="*80)
print("""
🎯 QUICK START GUIDE:

API Endpoints:
  • POST /ask          → Ask RAG a question (detailed response)
  • POST /ask_stream   → Stream response (real-time tokens)
  • POST /rebuild      → Manually trigger data sync now
  • GET /status        → Check system status & last update time
  • GET /health        → Simple health check

Example Queries:
  • "Có những sự kiện nào sắp tới?" → List upcoming events
  • "Sự kiện nào được review cao nhất?" → Find highly-rated events
  • "Hệ thống có vấn đề gì không?" (role=admin) → Check system errors

Auto-Update Schedule:
  • Runs every 1 hour automatically
  • Syncs data from database
  • Zero downtime - background process
  • Last Update: Check via GET /status

📞 From C# Backend:
  POST http://localhost:8000/ask
  {
    "question": "your question here",
    "role": "user",  // or "admin"
    "top_k": 10
  }

🔧 Configuration:
  • Update Interval: Edit IntervalTrigger(hours=1) in notebook
  • Change to: IntervalTrigger(minutes=5) for 5-minute updates
  • Change to: IntervalTrigger(hours=6) for 6-hour updates
""")


🎯 VERIFYING RAG AUTO-UPDATE SYSTEM

✅ [1/4] Checking API Health...
INFO:     127.0.0.1:60073 - "GET /health HTTP/1.1" 200 OK
    ✓ API Server: ONLINE
    ✓ Response: {'status': 'ok', 'model': 'RAG Event Consultant (Auto-Update Ready)', 'is_updating': False, 'last_update': '2026-03-05T04:32:54.890860'}

✅ [2/4] Getting System Status...
INFO:     127.0.0.1:60074 - "GET /status HTTP/1.1" 200 OK
    ✓ System Status: READY
    ✓ Last Updated: 2026-03-05T04:32:54.890860
    ✓ Total Updates: 3
    ✓ Auto-Update Interval: 1 hour (configurable)
    ✓ Currently Updating: False

✅ [3/4] Testing RAG Query (User Role)...
INFO:     127.0.0.1:60075 - "POST /ask HTTP/1.1" 200 OK
    ✓ Query: 'Có những sự kiện nào sắp tới?'
    ✓ Retrieved 4 relevant chunks
    ✓ Response Length: 778 characters
    ✓ Last System Update: 2026-03-05T04:32:54.890860

📝 Sample Answer (first 300 chars):
Có một số sự kiện sắp tới trong hệ thống AEMS. Dựa trên thông tin từ CONTEXT, tôi có thể xác định được hai sự kiện sắp tới

In [23]:
# ===== REINITIALIZE INDEXES FOR AUTO-UPDATE SYSTEM =====
print("🔄 Reinitializing indexes for periodic sync...")

try:
    # Force rebuild and save indexes
    from datetime import datetime
    start_time = datetime.now()
    print(f"[{start_time}] Building fresh indexes from database...")
    
    # Rebuild all indexes with feedback integration
    kb_index, log_index = build_all(days_logs=14)
    
    end_time = datetime.now()
    elapsed = (end_time - start_time).total_seconds()
    
    print(f"✅ Indexes rebuilt successfully in {elapsed:.1f} seconds")
    print(f"   KB Index: {kb_index.ntotal} vectors ({len(kb_chunks)} chunks)")
    print(f"   LOG Index: {log_index.ntotal} vectors ({len(log_chunks)} chunks)")
    
    # Mark initial update
    rag_state.mark_updated()
    
except Exception as e:
    print(f"⚠️ Rebuild warning (non-critical): {e}")
    print("   System will continue with existing indexes")


🔄 Reinitializing indexes for periodic sync...
[2026-03-05 04:32:11.711034] Building fresh indexes from database...
Building indexes with enhanced chunking + Feedback integration...


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_29260\2278150779.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  events = pd.read_sql("""
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_29260\2278150779.py:32: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  agenda = pd.read_sql("""
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_29260\2278150779.py:47: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  docs = pd.read_sql("""
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_29260\2278150779.py:58: UserWarning: pandas only supports SQLAlchemy co

Loaded: 2 events, 21 agenda rows, 0 docs, 2 locations, 0 feedbacks
Created 4 KB chunks from 2 events (with feedback)
Created 31 LOG chunks
✅ Saved kb/faiss_kb.index: 4 chunks
✅ Saved kb/faiss_log.index: 31 chunks
✅ Indexes rebuilt successfully in 8.7 seconds
   KB Index: 4 vectors (0 chunks)
   LOG Index: 31 vectors (0 chunks)


In [22]:
# ===== FIX: Define _build_and_save_index function =====
def build_index(texts: list):
    """Embed texts and create FAISS index."""
    embeddings = model.encode(texts)
    embeddings = embeddings / (  # Normalize for IP distance
        np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-10
    )
    dim = embeddings.shape[1]
    idx = faiss.IndexFlatIP(dim)
    idx.add(embeddings.astype('float32'))
    return idx

def _build_and_save_index(texts: list, metas: list, index_path: str, jsonl_path: str) -> 'faiss.IndexFlatIP':
    """Build and save FAISS index + metadata to JSONL."""
    if len(texts) == 0:
        raise ValueError(f"No texts to build index: {index_path}")
    
    idx = build_index(texts)
    
    os.makedirs("kb", exist_ok=True)
    faiss.write_index(idx, index_path)
    
    with open(jsonl_path, "w", encoding="utf-8") as f:
        for t, m in zip(texts, metas):
            f.write(json.dumps({"text": t, "meta": m}, ensure_ascii=False) + "\n")
    
    print(f"✅ Saved {index_path}: {len(texts)} chunks")
    return idx

print("✅ _build_and_save_index and build_index functions defined")

✅ _build_and_save_index and build_index functions defined


In [29]:
# ===== FIX: Define can_access function =====
def can_access(meta: dict, role: str):
    """Check if user role can access this chunk based on metadata."""
    role = (role or "user").lower()
    
    mtype = (meta.get("type") or "").lower()
    access = (meta.get("access") or "").lower()
    
    # log hệ thống: CHỈ admin
    if mtype == "system_log" or mtype == "system_log_summary":
        return role == "admin"
    
    # admin xem được hết
    if role == "admin":
        return True
    
    # user chỉ xem access=user
    return access == "user" or access == ""

print("✅ can_access function defined")

✅ can_access function defined


In [25]:
# ===== LOAD INDEXES FROM DISK =====
print("Loading FAISS indexes from disk...")

try:
    # Load KB index
    if os.path.exists("kb/faiss_kb.index"):
        kb_index = faiss.read_index("kb/faiss_kb.index")
        print(f"✅ Loaded KB index: {kb_index.ntotal} vectors")
    else:
        print("⚠️ KB index file not found")
        kb_index = None
    
    # Load LOG index
    if os.path.exists("kb/faiss_log.index"):
        log_index = faiss.read_index("kb/faiss_log.index")
        print(f"✅ Loaded LOG index: {log_index.ntotal} vectors")
    else:
        print("⚠️ LOG index file not found")
        log_index = None
    
    # Load chunks from JSONL
    kb_chunks = []
    if os.path.exists("kb/chunks_kb.jsonl"):
        with open("kb/chunks_kb.jsonl", "r", encoding="utf-8") as f:
            kb_chunks = [json.loads(line) for line in f if line.strip()]
        print(f"✅ Loaded {len(kb_chunks)} KB chunk texts")
    
    log_chunks = []
    if os.path.exists("kb/chunks_log.jsonl"):
        with open("kb/chunks_log.jsonl", "r", encoding="utf-8") as f:
            log_chunks = [json.loads(line) for line in f if line.strip()]
        print(f"✅ Loaded {len(log_chunks)} LOG chunk texts")
    
    print("\n✅ ALL INDEXES AND CHUNKS LOADED - READY FOR QUERIES")
    
except Exception as e:
    print(f"❌ Error loading indexes: {e}")

Loading FAISS indexes from disk...
✅ Loaded KB index: 4 vectors
✅ Loaded LOG index: 31 vectors
✅ Loaded 4 KB chunk texts
✅ Loaded 31 LOG chunk texts

✅ ALL INDEXES AND CHUNKS LOADED - READY FOR QUERIES


In [27]:
# ===== DIAGNOSTIC: Test retrieval directly =====
print("=" * 80)
print("🔍 DIAGNOSING RETRIEVAL SYSTEM")
print("=" * 80)

# Check variables
print(f"\n1. Check indexes in memory:")
print(f"   kb_index type: {type(kb_index)}")
if kb_index:
    print(f"   kb_index.ntotal: {kb_index.ntotal}")
else:
    print(f"   kb_index is None!")

print(f"   kb_chunks length: {len(kb_chunks)}")
if kb_chunks:
    print(f"   First chunk preview: {str(kb_chunks[0])[:100]}")

# Test embedding a query
test_q = "Có những sự kiện nào sắp tới?"
print(f"\n2. Embed test query: '{test_q}'")
try:
    q_emb = model.encode(test_q)
    q_emb = q_emb / (np.linalg.norm(q_emb) + 1e-10)
    print(f"   ✓ Query embedding shape: {q_emb.shape}")
    print(f"   ✓ Query embedding (first 5 dims): {q_emb[:5]}")
except Exception as e:
    print(f"   ✗ Error encoding query: {e}")

# Test direct index search
print(f"\n3. Search KB index directly:")
try:
    if kb_index and kb_index.ntotal > 0:
        q_emb_batch = q_emb[np.newaxis, :].astype('float32')  # Shape: (1, 384)
        scores, indices = kb_index.search(q_emb_batch, k=min(5, kb_index.ntotal))
        print(f"   ✓ Search returned scores: {scores[0]}")
        print(f"   ✓ Search returned indices: {indices[0]}")
        print(f"   ✓ Top chunk: {kb_chunks[indices[0][0]].get('text', '')[:100] if indices[0][0] < len(kb_chunks) else 'INDEX OUT OF RANGE'}")
    else:
        print(f"   ✗ KB index is None or empty!")
except Exception as e:
    print(f"   ✗ Error searching: {e}")
    import traceback
    traceback.print_exc()

print(f"\n4. Check retrieve functions:")
print(f"   'retrieve' function defined: {'retrieve' in globals()}")
print(f"   'retrieve_with_fallback' function defined: {'retrieve_with_fallback' in globals()}")

# Test via the actual retrieve function
print(f"\n5. Test retrieve_with_fallback():")
try:
    hits = retrieve_with_fallback(test_q, top_k=5, role="user")
    print(f"   ✓ Retrieved {len(hits)} chunks")
    if hits:
        print(f"   Top hit score: {hits[0].get('score', 'N/A')}")
        print(f"   Top hit text: {hits[0].get('text', '')[:100]}")
except Exception as e:
    print(f"   ✗ Error: {e}")
    import traceback
    traceback.print_exc()


🔍 DIAGNOSING RETRIEVAL SYSTEM

1. Check indexes in memory:
   kb_index type: <class 'faiss.swigfaiss_avx2.IndexFlatIP'>
   kb_index.ntotal: 4
   kb_chunks length: 4
   First chunk preview: {'text': 'SỰ KIỆN: AI Workshop 2026\n\nMÔ TẢ:\nWorkshop giới thiệu về AI và Machine Learning.\n\nTHÔ

2. Embed test query: 'Có những sự kiện nào sắp tới?'
   ✓ Query embedding shape: (384,)
   ✓ Query embedding (first 5 dims): [-0.06237933  0.07910089 -0.01139266  0.03276878 -0.04544594]

3. Search KB index directly:
   ✓ Search returned scores: [0.4540053  0.37370566 0.3616863  0.3437673 ]
   ✓ Search returned indices: [2 0 3 1]
   ✓ Top chunk: SỰ KIỆN: Software Testing

MÔ TẢ:
Khóa học chuyên sâu

THÔNG TIN CƠ BẢN:
- Bắt đầu: 2026-04-05 13:00

4. Check retrieve functions:
   'retrieve' function defined: True
   'retrieve_with_fallback' function defined: True

5. Test retrieve_with_fallback():
   ✓ Retrieved 4 chunks
   Top hit score: 0.4456957280635834
   Top hit text: SỰ KIỆN: Software Testing

MÔ

In [30]:
# ===== TEST API RESPONSE FORMAT =====
print("\n6. Test actual API /ask endpoint response:")

try:
    response = requests.post(
        "http://127.0.0.1:8000/ask",
        json={
            "question": "Có những sự kiện nào sắp tới?",
            "role": "user",
            "top_k": 10
        }
    )
    print(f"   Status code: {response.status_code}")
    data = response.json()
    print(f"   Response keys: {list(data.keys())}")
    print(f"   Question: {data.get('question')}")
    print(f"   Answer length: {len(data.get('answer', ''))}")
    print(f"   Total hits: {data.get('total_hits', 'N/A')}")
    print(f"   Sources count: {len(data.get('sources', []))}")
    if data.get('error'):
        print(f"   Error: {data.get('error')}")
    print(f"\n   Response (first 800 chars):")
    print(str(data)[:800])
except Exception as e:
    print(f"   ✗ Error: {e}")
    import traceback
    traceback.print_exc()


6. Test actual API /ask endpoint response:
INFO:     127.0.0.1:53931 - "POST /ask HTTP/1.1" 200 OK
   Status code: 200
   Response keys: ['question', 'answer', 'sources', 'total_hits', 'last_updated']
   Question: Có những sự kiện nào sắp tới?
   Answer length: 1021
   Total hits: 4
   Sources count: 4

   Response (first 800 chars):
{'question': 'Có những sự kiện nào sắp tới?', 'answer': 'Cảm ơn bạn đã cung cấp thông tin hệ thống. Dựa trên thông tin được cung cấp, có hai sự kiện sắp tới:\n\n1. **Software Testing**:\n - Bắt đầu: 2026-04-05 13:00:00\n - Kết thúc: 2026-04-05 17:00:00\n - Hình thức: None\n - Sức chứa: 80 người\n - Loại sự kiện: Bootcamp\n - Phòng ban: Software Engineering\n - Chủ đề: \n - Người tổ chức: N/A\n - Đặt cọc: 200000.0\n\nSự kiện này là một khóa học chuyên sâu về software testing, được tổ chức vào ngày 5 tháng 4 năm 2026. Thông tin chi tiết về địa điểm và chủ đề của sự kiện này không được cung cấp trong hệ thống.\n\n2. **AI Workshop 2026**:\n - Bắt đầu: 2026-03

In [1]:
import os
import glob
from dotenv import load_dotenv
import gradio as gr
import functools
from concurrent.futures import ThreadPoolExecutor
import time
import pyodbc
import pandas as pd
import os, json
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
import requests
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from tqdm import tqdm
import nest_asyncio
import uvicorn

In [2]:
import pyodbc
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Get credentials from environment variables (SECURE)
SERVER = os.getenv("DB_SERVER", "tcp:swp391-sp26-server.database.windows.net")
DATABASE = os.getenv("DB_NAME", "SWP391_SP26_Group1")
USERNAME = os.getenv("DB_USER", "SWP391_SP26_Group1")
PASSWORD = os.getenv("DB_PASSWORD", "Team11111111")

if not PASSWORD:
    print("⚠️ WARNING: DB_PASSWORD not set in .env file!")

def get_conn():
    conn_str = (
        "DRIVER={ODBC Driver 18 for SQL Server};"
        f"SERVER={SERVER},1433;"
        f"DATABASE={DATABASE};"
        f"UID={USERNAME};"
        f"PWD={PASSWORD};"
        "Encrypt=yes;"
        "TrustServerCertificate=no;"
        "Connection Timeout=30;"
    )
    return pyodbc.connect(conn_str)

# Test connection
try:
    conn = get_conn()
    cursor = conn.cursor()
    cursor.execute("SELECT TOP 5 * FROM dbo.SystemErrorLog;")
    for row in cursor.fetchall():
        print(row)
    conn.close()
except Exception as e:
    print(f"❌ Connection error: {e}")


('00074562-9970-49ea-ac25-25dda6782f21', 500, 'ArgumentException', "The 'ClientId' option must be provided. (Parameter 'ClientId') | Inner Cause: The 'ClientId' option must be provided. (Parameter 'ClientId')", '   at Microsoft.AspNetCore.Authentication.OAuth.OAuthOptions.Validate()\r\n   at Microsoft.AspNetCore.Authentication.RemoteAuthenticationOptions.Validate(String scheme)\r\n   at Microsoft.AspNetCore.Authentication.AuthenticationBuilder.<>c__DisplayClass4_0`2.<AddSchemeHelper>b__1(TOptions o)\r\n   at Microsoft.Extensions.Options.ValidateOptions`1.Validate(String name, TOptions options)\r\n   at Microsoft.Extensions.Options.OptionsFactory`1.Create(String name)\r\n   at Microsoft.Extensions.Options.OptionsMonitor`1.<>c.<Get>b__10_0(String name, IOptionsFactory`1 factory)\r\n   at Microsoft.Extensions.Options.OptionsCache`1.<>c__DisplayClass3_1`1.<GetOrAdd>b__2()\r\n   at System.Lazy`1.ViaFactory(LazyThreadSafetyMode mode)\r\n--- End of stack trace from previous location ---\r\n  

In [3]:
from concurrent.futures import ThreadPoolExecutor

In [4]:
#Knowledge Base

In [3]:


def load_events_bundle():
    """Load events, agendas, docs, locations, AND FEEDBACK from database with full details."""
    
    # 1) Event - simplified queries to avoid schema issues
    events = pd.read_sql("""
        SELECT
            e.Id AS EventId,
            e.Title,
            e.Description,
            e.StartTime,
            e.EndTime,
            e.Status,
            e.Mode,
            e.MeetingUrl,
            e.MaxCapacity,
            e.Type,
            e.IsDepositRequired,
            e.DepositAmount,
            e.DepartmentId,
            e.SemesterId,
            e.LocationId,
            e.TopicId,
            e.OrganizerId,
            e.PublishedAt,
            e.UpdatedAt
        FROM Event e
        WHERE (e.DeletedAt IS NULL OR e.DeletedAt = '') 
          AND e.Status IN ('Published')  
    """, get_conn())

    # 2) EventAgenda
    agenda = pd.read_sql("""
        SELECT
            a.Id AS AgendaId,
            a.EventId,
            a.SessionName,
            a.Description AS SessionDescription,
            a.SpeakerName,
            a.StartTime AS SessionStart,
            a.EndTime AS SessionEnd,
            a.Location AS SessionLocation
        FROM EventAgenda a
        WHERE (a.DeletedAt IS NULL)
    """, get_conn())

    # 3) EventDocuments
    docs = pd.read_sql("""
        SELECT
            d.Id AS DocId,
            d.EventId,
            d.Name AS DocTitle,
            d.Url
        FROM EventDocument d
        WHERE (d.DeletedAt IS NULL)
    """, get_conn())
    
    # 4) Locations
    locations = pd.read_sql("""
        SELECT
           l.Id        AS LocationId,
           l.Name      AS LocationName,
           l.Address,
           l.Capacity,
           l.Status,
           l.Type,
           l.Description,
           l.CreatedAt,
           l.UpdatedAt
        FROM Locations l
        WHERE (l.DeletedAt IS NULL)
    """, get_conn())
    
    # 5) FEEDBACK - Chi tiết đánh giá sự kiện
    feedback = pd.read_sql("""
        SELECT
            f.Id AS FeedbackId,
            f.EventId,
            f.Rating,
            f.Comment,
            f.CreatedAt,
            f.UpdatedAt
        FROM Feedback f
        WHERE (f.DeletedAt IS NULL)
    """, get_conn())
    
    # Enrich events with department & organizer names (safe)
    try:
        depts = pd.read_sql("SELECT Id, Name FROM Department WHERE DeletedAt IS NULL", get_conn())
        events = events.merge(depts, left_on='DepartmentId', right_on='Id', how='left', suffixes=('', '_dept'))
        events = events.rename(columns={'Name': 'DepartmentName'})
    except:
        events['DepartmentName'] = 'N/A'
    
    try:
        organizers = pd.read_sql("SELECT Id, FullName FROM StaffProfile WHERE DeletedAt IS NULL", get_conn())
        events = events.merge(organizers, left_on='OrganizerId', right_on='Id', how='left', suffixes=('', '_org'))
        events = events.rename(columns={'FullName': 'OrganizerName'})
    except:
        events['OrganizerName'] = 'N/A'

    return events, agenda, docs, locations, feedback

def load_logs(days=14):
    """Load system error logs from database."""
    logs = pd.read_sql("""
        SELECT
            sel.Id AS LogId,
            sel.StatusCode,
            sel.ExceptionType,
            sel.ExceptionMessage,
            sel.StackTrace,
            sel.Source,
            sel.UserId,
            sel.CreatedAt
        FROM SystemErrorLog sel
        WHERE sel.CreatedAt IS NOT NULL
    """, get_conn())
    return logs


In [6]:
#get role DB

In [49]:
def get_user_role(db, user_id: str) -> str:
    row = db.execute("""
        SELECT r.RoleName
        FROM [User] u
        JOIN Role r ON u.RoleId = r.Id
        WHERE u.Id = :user_id
    """, {"user_id": user_id}).fetchone()

    return row[0].lower() if row else "user"


In [50]:
def can_access(meta: dict, role: str):
    role = (role or "user").lower()

    mtype = (meta.get("type") or "").lower()
    access = (meta.get("access") or "").lower()

    # log hệ thống: CHỈ admin
    if mtype == "system_log":
        return role == "admin"

    # admin xem được hết
    if role == "admin":
        return True

    # user chỉ xem access=user
    return access == "user"


In [9]:
import sys
print(sys.executable)


d:\PROJECT_SWP_SP26\Python\.venv-rag\Scripts\python.exe


In [7]:
def build_event_text(event_row, agenda_rows, doc_rows, location_details=None):
    """Build comprehensive event knowledge base text with detailed consultant information."""
    lines = []
    
    # 1) EVENT BASIC INFO
    lines.append("="*60)
    lines.append("THÔNG TIN SỰ KIỆN")
    lines.append("="*60)
    lines.append(f"Tên sự kiện: {event_row.get('Title','')}")
    lines.append(f"Mô tả: {event_row.get('Description','')}")
    lines.append("")
    
    # 2) EVENT TIMING & LOGISTICS
    lines.append("THỜI GIAN & ĐỊA ĐIỂM")
    lines.append("-"*60)
    start = event_row.get('StartTime','')
    end = event_row.get('EndTime','')
    lines.append(f"Bắt đầu: {start}")
    lines.append(f"Kết thúc: {end}")
    
    # Mode & Meeting info
    mode = event_row.get('Mode','')
    if mode:
        lines.append(f"Hình thức: {mode}")
        if mode.lower() in ['online', 'hybrid'] and event_row.get('MeetingUrl'):
            lines.append(f"Link cuộc họp: {event_row.get('MeetingUrl')}")
    
    max_cap = event_row.get('MaxCapacity', '')
    if max_cap:
        lines.append(f"Sức chứa tối đa: {max_cap} người")
    lines.append("")
    
    # 3) EVENT DETAILS
    lines.append("CHI TIẾT SỰ KIỆN")
    lines.append("-"*60)
    if event_row.get('Type'):
        lines.append(f"Loại sự kiện: {event_row.get('Type')}")
    if event_row.get('DepartmentName'):
        lines.append(f"Phòng ban tổ chức: {event_row.get('DepartmentName')}")
    if event_row.get('TopicName'):
        lines.append(f"Chủ đề: {event_row.get('TopicName')}")
    if event_row.get('OrganizerName'):
        lines.append(f"Người tổ chức: {event_row.get('OrganizerName')}")
    
    # Deposit info
    if event_row.get('IsDepositRequired'):
        deposit = event_row.get('DepositAmount', 'TBD')
        lines.append(f"Yêu cầu đặt cọc: Có (Số tiền: {deposit})")
    lines.append("")
    
    # 4) AGENDA DETAILS
    if len(agenda_rows) > 0:
        lines.append("CHƯƠNG TRÌNH CHI TIẾT")
        lines.append("-"*60)
        for idx, (_, a) in enumerate(agenda_rows.iterrows(), 1):
            lines.append(f"\nPhiên {idx}: {a.get('SessionName','')}")
            if a.get('SpeakerName'):
                lines.append(f"  Diễn giả: {a.get('SpeakerName')}")
            if a.get('SessionStart') or a.get('SessionEnd'):
                lines.append(f"  Thời gian: {a.get('SessionStart','')} - {a.get('SessionEnd','')}")
            if a.get('SessionLocation'):
                lines.append(f"  Địa điểm: {a.get('SessionLocation')}")
            if a.get('SessionDescription'):
                lines.append(f"  Mô tả: {a.get('SessionDescription')}")
        lines.append("")
    
    # 5) DOCUMENTS & RESOURCES
    if len(doc_rows) > 0:
        lines.append("TÀI LIỆU & TÀI NGUYÊN")
        lines.append("-"*60)
        for idx, (_, d) in enumerate(doc_rows.iterrows(), 1):
            lines.append(f"{idx}. {d.get('DocTitle','')}")
            if d.get("Url"):
                lines.append(f"   URL: {d.get('Url')}")
        lines.append("")
    
    # 6) LOCATION DETAILS
    if location_details is not None and isinstance(location_details, dict) and len(location_details) > 0:
        lines.append("ĐỊA ĐIỂM CHI TIẾT")
        lines.append("-"*60)
        lines.append(f"Tên địa điểm: {location_details.get('LocationName','')}")
        if location_details.get("Address"):
            lines.append(f"Địa chỉ: {location_details.get('Address')}")
        if location_details.get("Capacity"):
            lines.append(f"Sức chứa: {location_details.get('Capacity')} người")
        if location_details.get("Type"):
            lines.append(f"Loại địa điểm: {location_details.get('Type')}")
        if location_details.get("Description"):
            lines.append(f"Mô tả: {location_details.get('Description')}")
        lines.append("")

    return "\n".join(lines).strip()


In [8]:
# Build KB logs (admin) - optimized: shorten stacktrace + enable roll-up summaries
import re

def _short_stack(s: str, max_lines: int = 15, max_chars: int = 1200) -> str:
    s = (s or "").strip()
    if not s:
        return ""
    lines = s.splitlines()[:max_lines]
    out = "\n".join(lines)
    return out[:max_chars]

def build_systemlog_text(r):
    return "\n".join([
        "SYSTEM ERROR LOG",
        f"StatusCode: {r.get('StatusCode','')}",
        f"ExceptionType: {r.get('ExceptionType','')}",
        f"ExceptionMessage: {r.get('ExceptionMessage','')}",
        f"Source: {r.get('Source','')}",
        f"UserId: {r.get('UserId','')}",
        f"CreatedAt: {r.get('CreatedAt','')}",
        f"StackTrace: {_short_stack(r.get('StackTrace',''))}",
    ])

# ---- Log roll-up (IMPORTANT for admin: prevents log-chunk flooding) ----
def _normalize_msg(s: str) -> str:
    s = (s or "").lower()
    # replace guids/hashes/numbers that cause duplicates to look different
    s = re.sub(r"\b[0-9a-f]{8,}\b", "<id>", s)
    s = re.sub(r"\b\d+\b", "<n>", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s[:400]

def rollup_logs(logs_df: pd.DataFrame) -> pd.DataFrame:
    """Group similar errors into summary rows (Count/FirstAt/LastAt)."""
    if logs_df is None or len(logs_df) == 0:
        return logs_df

    df = logs_df.copy()
    df["MsgNorm"] = df["ExceptionMessage"].apply(_normalize_msg)
    df["Sig"] = (
        df["ExceptionType"].fillna("").astype(str) + "|" +
        df["Source"].fillna("").astype(str) + "|" +
        df["StatusCode"].fillna("").astype(str) + "|" +
        df["MsgNorm"].fillna("").astype(str)
    )

    g = df.groupby("Sig", as_index=False).agg(
        Count=("Sig", "size"),
        FirstAt=("CreatedAt", "min"),
        LastAt=("CreatedAt", "max"),
        ExceptionType=("ExceptionType", "first"),
        ExceptionMessage=("ExceptionMessage", "first"),
        Source=("Source", "first"),
        StatusCode=("StatusCode", "first"),
    )
    return g.sort_values(["Count", "LastAt"], ascending=[False, False])


In [13]:
def build_all(days_logs: int = 14):
    """Build both KB and LOG indexes from database with smart chunking + FEEDBACK."""
    print("Building indexes with enhanced chunking + Feedback integration...")

    events, agenda, docs, locations, feedback = load_events_bundle()
    logs = load_logs(days=days_logs)
    
    print(f"Loaded: {len(events)} events, {len(agenda)} agenda rows, {len(docs)} docs, {len(locations)} locations, {len(feedback)} feedbacks")

    # Tính rating trung bình cho mỗi event
    feedback_stats = {}
    if len(feedback) > 0:
        fb_grouped = feedback.groupby('EventId').agg({
            'Rating': ['mean', 'count', 'min', 'max']
        }).reset_index()
        fb_grouped.columns = ['EventId', 'AvgRating', 'ReviewCount', 'MinRating', 'MaxRating']
        for _, row in fb_grouped.iterrows():
            feedback_stats[row['EventId']] = {
                'avg_rating': round(float(row['AvgRating']), 2),
                'review_count': int(row['ReviewCount']),
                'min_rating': int(row['MinRating']),
                'max_rating': int(row['MaxRating'])
            }

    # Build KB chunks with SMART CHUNKING (1 event = multiple chunks)
    kb_chunks_data = []
    kb_meta_data = []

    for _, event_row in events.iterrows():
        event_id = event_row.get('EventId','')
        
        # Get related agenda & docs for this event
        event_agenda = agenda[agenda['EventId'] == event_id]
        event_docs = docs[docs['EventId'] == event_id]
        event_feedback = feedback[feedback['EventId'] == event_id]
        location_id = event_row.get('LocationId','')
        location_details = None
        if location_id:
            loc_match = locations[locations['LocationId'] == location_id]
            if len(loc_match) > 0:
                location_details = loc_match.iloc[0].to_dict()
        
        # Get feedback stats
        fb_stat = feedback_stats.get(event_id, {})
        
        # ==== CHUNK 1: Event Overview & Basic Info + FEEDBACK RATING ====
        overview_text = f"""SỰ KIỆN: {event_row.get('Title','UNKNOWN')}

MÔ TẢ:
{event_row.get('Description','')}

THÔNG TIN CƠ BẢN:
- Bắt đầu: {event_row.get('StartTime','')}
- Kết thúc: {event_row.get('EndTime','')}
- Hình thức: {event_row.get('Mode','Offline')}
- Sức chứa: {event_row.get('MaxCapacity','TBD')} người
- Loại sự kiện: {event_row.get('Type','')}
- Phòng ban: {event_row.get('DepartmentName','')}
- Chủ đề: {event_row.get('TopicName','')}
- Người tổ chức: {event_row.get('OrganizerName','')}
"""
        if event_row.get('MeetingUrl'):
            overview_text += f"- Link cuộc họp: {event_row.get('MeetingUrl')}\n"
        if event_row.get('IsDepositRequired'):
            overview_text += f"- Đặt cọc: {event_row.get('DepositAmount','TBD')}\n"
        
        # Thêm feedback info
        if fb_stat:
            overview_text += f"\nĐánh giá từ người tham gia:\n"
            overview_text += f"- Điểm trung bình: {fb_stat.get('avg_rating', 0)}/5 sao\n"
            overview_text += f"- Số lượng đánh giá: {fb_stat.get('review_count', 0)} reviews\n"
            overview_text += f"- Điểm cao nhất: {fb_stat.get('max_rating', 0)}/5\n"
            overview_text += f"- Điểm thấp nhất: {fb_stat.get('min_rating', 0)}/5\n"
        
        kb_chunks_data.append(overview_text.strip())
        kb_meta_data.append({
            "type": "event_overview",
            "access": "user",
            "event_id": event_id,
            "event_title": event_row.get('Title',''),
            "created_at": str(event_row.get('PublishedAt','')),
            "mode": event_row.get('Mode',''),
            "avg_rating": fb_stat.get('avg_rating', 0),
            "review_count": fb_stat.get('review_count', 0),
            "source_kind": "event_basic"
        })
        
        # ==== CHUNK 2: Agenda Details (if exists) ====
        if len(event_agenda) > 0:
            agenda_text = f"CHƯƠNG TRÌNH CHI TIẾT - {event_row.get('Title','')}\n\n"
            for _, session in event_agenda.iterrows():
                agenda_text += f"""SESSION: {session.get('SessionName','')}
Diễn giả: {session.get('SpeakerName','')}
Thời gian: {session.get('SessionStart','')} đến {session.get('SessionEnd','')}
Địa điểm phiên: {session.get('SessionLocation','')}
Nội dung: {session.get('SessionDescription','')}

"""
            kb_chunks_data.append(agenda_text.strip())
            kb_meta_data.append({
                "type": "event_agenda",
                "access": "user",
                "event_id": event_id,
                "event_title": event_row.get('Title',''),
                "session_count": len(event_agenda),
                "created_at": str(event_row.get('PublishedAt','')),
                "source_kind": "event_agenda"
            })
        
        # ==== CHUNK 3: Location Details (if exists) ====
        if location_details:
            location_text = f"""ĐỊA ĐIỂM TỔNG CHỮ KIỆN: {event_row.get('Title','')}

Tên địa điểm: {location_details.get('LocationName','')}
Địa chỉ: {location_details.get('Address','')}
Sức chứa: {location_details.get('Capacity','')} người
Loại: {location_details.get('Type','')}
Trạng thái: {location_details.get('Status','')}
Mô tả: {location_details.get('Description','')}
"""
            kb_chunks_data.append(location_text.strip())
            kb_meta_data.append({
                "type": "event_location",
                "access": "user",
                "event_id": event_id,
                "event_title": event_row.get('Title',''),
                "location_id": location_id,
                "location_name": location_details.get('LocationName',''),
                "created_at": str(event_row.get('PublishedAt','')),
                "source_kind": "event_location"
            })
        
        # ==== CHUNK 4: Documents & Resources (if exists) ====
        if len(event_docs) > 0:
            docs_text = f"""TÀI LIỆU & RESOURCES - {event_row.get('Title','')}

"""
            for _, doc in event_docs.iterrows():
                docs_text += f"- {doc.get('DocTitle','')}\n"
                if doc.get('Url'):
                    docs_text += f"  URL: {doc.get('Url')}\n"
            
            kb_chunks_data.append(docs_text.strip())
            kb_meta_data.append({
                "type": "event_documents",
                "access": "user",
                "event_id": event_id,
                "event_title": event_row.get('Title',''),
                "doc_count": len(event_docs),
                "created_at": str(event_row.get('PublishedAt','')),
                "source_kind": "event_docs"
            })
        
        # ==== CHUNK 5: Feedback Reviews (if exists) ====
        if len(event_feedback) > 0:
            feedback_text = f"""ĐÁNH GIÁ TỪ NGƯỜI THAM GIA - {event_row.get('Title','')}

Điểm trung bình: {fb_stat.get('avg_rating', 0)}/5 sao ({fb_stat.get('review_count', 0)} đánh giá)

CHI TIẾT ĐÁNH GIÁ:
"""
            for idx, (_, fb) in enumerate(event_feedback.head(5).iterrows(), 1):  # Top 5 reviews
                feedback_text += f"\nĐánh giá {idx}:\n"
                feedback_text += f"- Điểm: {fb.get('Rating', 0)}/5 sao\n"
                if fb.get('Comment'):
                    feedback_text += f"- Nhận xét: {fb.get('Comment')}\n"
                feedback_text += f"- Ngày: {fb.get('CreatedAt', '')}\n"
            
            kb_chunks_data.append(feedback_text.strip())
            kb_meta_data.append({
                "type": "event_feedback",
                "access": "user",
                "event_id": event_id,
                "event_title": event_row.get('Title',''),
                "feedback_count": len(event_feedback),
                "avg_rating": fb_stat.get('avg_rating', 0),
                "created_at": str(event_row.get('PublishedAt','')),
                "source_kind": "event_feedback"
            })

    print(f"Created {len(kb_chunks_data)} KB chunks from {len(events)} events (with feedback)")

    # Build LOG chunks (rollup + detailed)
    log_chunks_data = []
    log_meta_data = []

    log_sum = rollup_logs(logs) if logs is not None and len(logs) > 0 else None
    
    if log_sum is not None and len(log_sum) > 0:
        for _, r in log_sum.iterrows():
            text = "\n".join([
                "SYSTEM ERROR SUMMARY",
                f"StatusCode: {r.get('StatusCode','')}",
                f"ExceptionType: {r.get('ExceptionType','')}",
                f"Source: {r.get('Source','')}",
                f"Message: {r.get('ExceptionMessage','')}",
                f"Count: {r.get('Count','')}",
                f"FirstAt: {r.get('FirstAt','')}",
                f"LastAt: {r.get('LastAt','')}",
            ])
            log_chunks_data.append(text)
            log_meta_data.append({
                "type": "system_log_summary",
                "access": "admin",
                "status_code": str(r.get("StatusCode", "")),
                "exception_type": str(r.get("ExceptionType", "")),
                "source": str(r.get("Source", "")),
                "count": int(r.get("Count", 0)) if str(r.get("Count","")).isdigit() else r.get("Count", 0),
                "first_at": str(r.get("FirstAt", "")),
                "last_at": str(r.get("LastAt", "")),
                "source_kind": "log_rollup"
            })

    print(f"Created {len(log_chunks_data)} LOG chunks")

    # Build and save indexes
    kb_index = _build_and_save_index(
        kb_chunks_data, kb_meta_data,
        index_path="kb/faiss_kb.index",
        jsonl_path="kb/chunks_kb.jsonl"
    )

    log_index = None
    if log_chunks_data:
        log_index = _build_and_save_index(
            log_chunks_data, log_meta_data,
            index_path="kb/faiss_log.index",
            jsonl_path="kb/chunks_log.jsonl"
        )

    return kb_index, log_index


print("Starting main build process...")
kb_index, log_index = build_all(days_logs=14)
print("✅ Build complete!")


Starting main build process...
Building indexes with enhanced chunking + Feedback integration...


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_29260\2278150779.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  events = pd.read_sql("""
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_29260\2278150779.py:32: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  agenda = pd.read_sql("""
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_29260\2278150779.py:47: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  docs = pd.read_sql("""
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_29260\2278150779.py:58: UserWarning: pandas only supports SQLAlchemy co

Loaded: 2 events, 21 agenda rows, 0 docs, 2 locations, 0 feedbacks
Created 4 KB chunks from 2 events (with feedback)
Created 31 LOG chunks


NameError: name '_build_and_save_index' is not defined

In [10]:
#chunking context
from typing import List

def chunk_text(text: str, max_chars: int = 3000, overlap: int = 250) -> List[str]:
    """Split text into overlapping chunks."""
    chunks = []
    i = 0
    while i < len(text):
        j = min(len(text), i + max_chars)
        chunks.append(text[i:j])
        i = j - overlap
        if i < 0:
            i = 0
        if j == len(text):
            break
    return chunks

# ===== LOAD EMBEDDING MODEL =====
print("Loading SentenceTransformer model...")
try:
    model = SentenceTransformer('all-MiniLM-L6-v2')  # Fast & lightweight
    print(f"✅ Model loaded: {model.get_sentence_embedding_dimension()} dimensions")
except Exception as e:
    print(f"❌ Model loading error: {e}")
    model = None

# ===== BUILD INDEX FUNCTION =====
def build_index(texts: List[str]) -> faiss.IndexFlatIP:
    """Build FAISS index from text chunks."""
    if not texts:
        raise ValueError("No texts provided to build index")
    
    if model is None:
        raise RuntimeError("Model not loaded")
    
    print(f"Encoding {len(texts)} texts...")
    passages = [f"passage: {t}" for t in texts]
    embs = model.encode(passages, normalize_embeddings=True, show_progress_bar=True)
    embs = np.asarray(embs, dtype="float32")
    
    if embs.ndim == 1:
        embs = embs.reshape(1, -1)
    
    dim = embs.shape[1]
    idx = faiss.IndexFlatIP(dim)
    idx.add(embs)
    
    print(f"✅ Index built: {len(texts)} chunks, {dim} dimensions")
    return idx

# ===== GROQ API ENDPOINT =====
GROQ_URL = "https://api.groq.com/openai/v1/chat/completions"


Loading SentenceTransformer model...


d:\PROJECT_SWP_SP26\Python\.venv-rag\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


✅ Model loaded: 384 dimensions


In [14]:
import os

PROJECT_DIR = r"D:\PROJECT_SWP_SP26\Python"   # đúng folder project của bạn
os.chdir(PROJECT_DIR)

KB_DIR = os.path.join(PROJECT_DIR, "kb")
os.makedirs(KB_DIR, exist_ok=True)

print("CWD =", os.getcwd())
print("KB exists?", os.path.exists(KB_DIR))
print("KB files:", os.listdir(KB_DIR))


CWD = D:\PROJECT_SWP_SP26\Python
KB exists? True
KB files: ['chunks_kb.jsonl', 'chunks_log.jsonl', 'faiss_kb.index', 'faiss_log.index']


In [15]:
import os, faiss

kb_path  = os.path.join(KB_DIR, "faiss_kb.index")
log_path = os.path.join(KB_DIR, "faiss_log.index")

if not os.path.exists(kb_path) or not os.path.exists(log_path):
    print("⚠️ Index not found → building...")
    kb_index, log_index = build_all(days_logs=14)
else:
    print("✅ Loading existing index...")
    kb_index  = faiss.read_index(kb_path)
    log_index = faiss.read_index(log_path)

print("KB ntotal =", kb_index.ntotal)
print("LOG ntotal =", log_index.ntotal)


✅ Loading existing index...
KB ntotal = 1
LOG ntotal = 35


In [16]:
def _build_and_save_index(texts: List[str], metas: list, index_path: str, jsonl_path: str) -> faiss.IndexFlatIP:
    """Build and save FAISS index + metadata to JSONL."""
    if len(texts) == 0:
        raise ValueError(f"No texts to build index: {index_path}")

    idx = build_index(texts)

    os.makedirs("kb", exist_ok=True)
    faiss.write_index(idx, index_path)

    with open(jsonl_path, "w", encoding="utf-8") as f:
        for t, m in zip(texts, metas):
            f.write(json.dumps({"text": t, "meta": m}, ensure_ascii=False) + "\n")

    print(f"✅ Saved {index_path}: {len(texts)} chunks")
    return idx


def main(days_logs_rollup: int = 14):
    """Main pipeline: load data → chunk → encode → index."""
    print("📊 Loading data from database...")
    events, agenda, docs, locations = load_events_bundle()
    logs = load_logs(days=days_logs_rollup)

    # Filter logs by date
    if "CreatedAt" in logs.columns:
        try:
            cutoff = pd.Timestamp.utcnow() - pd.Timedelta(days=days_logs_rollup)
            logs = logs[logs["CreatedAt"] >= cutoff]
        except Exception as e:
            print(f"⚠️ Date filtering error: {e}")

    print(f"Events: {len(events)} | Logs: {len(logs)}")

    # 1) Knowledge base chunks (events/docs)
    print("📝 Building KB chunks...")
    kb_chunks_data, kb_meta_data = [], []
    for _, e in events.iterrows():
        eid = e["EventId"]
        a_rows = agenda[agenda["EventId"] == eid]
        d_rows = docs[docs["EventId"] == eid]
        doc_text = build_event_text(e, a_rows, d_rows, locations_rows=locations)
        chunks = chunk_text(doc_text)

        for idx, ch in enumerate(chunks):
            ch = (ch or "").strip()
            if not ch:
                continue

            kb_chunks_data.append(ch)
            kb_meta_data.append({
                "type": "event",
                "access": "user",
                "event_id": str(eid),
                "chunk_id": idx,
                "department_id": str(e.get("DepartmentId", "")),
                "semester_id": str(e.get("SemesterId", "")),
                "status": str(e.get("Status", "")),
                "updated_at": str(e.get("UpdatedAt", "")),
                "source": "event_bundle"
            })

    # 2) Admin log chunks (roll-up summary)
    print("📋 Building LOG summary chunks...")
    log_sum = rollup_logs(logs)

    log_chunks_data, log_meta_data = [], []
    if log_sum is not None and len(log_sum) > 0:
        for _, r in log_sum.iterrows():
            text = "\n".join([
                "SYSTEM ERROR SUMMARY",
                f"StatusCode: {r.get('StatusCode','')}",
                f"ExceptionType: {r.get('ExceptionType','')}",
                f"Source: {r.get('Source','')}",
                f"Message: {r.get('ExceptionMessage','')}",
                f"Count: {r.get('Count','')}",
                f"FirstAt: {r.get('FirstAt','')}",
                f"LastAt: {r.get('LastAt','')}",
            ])
            log_chunks_data.append(text)
            log_meta_data.append({
                "type": "system_log_summary",
                "access": "admin",
                "status_code": str(r.get("StatusCode", "")),
                "exception_type": str(r.get("ExceptionType", "")),
                "source": str(r.get("Source", "")),
                "count": int(r.get("Count", 0)) if str(r.get("Count","")).isdigit() else r.get("Count", 0),
                "first_at": str(r.get("FirstAt", "")),
                "last_at": str(r.get("LastAt", "")),
                "source_kind": "log_rollup"
            })

    print(f"KB chunks: {len(kb_chunks_data)} | LOG chunks: {len(log_chunks_data)}")

    # Build and save indexes
    kb_index = _build_and_save_index(
        kb_chunks_data, kb_meta_data,
        index_path="kb/faiss_kb.index",
        jsonl_path="kb/chunks_kb.jsonl"
    )

    log_index = None
    if log_chunks_data:
        log_index = _build_and_save_index(
            log_chunks_data, log_meta_data,
            index_path="kb/faiss_log.index",
            jsonl_path="kb/chunks_log.jsonl"
        )

    return kb_index, log_index


print("Starting main build process...")
kb_index, log_index = main(days_logs_rollup=14)
print("✅ Build complete!")


Starting main build process...
📊 Loading data from database...


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_28996\404720421.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  events = pd.read_sql("""
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_28996\404720421.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  agenda = pd.read_sql("""
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_28996\404720421.py:36: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  docs = pd.read_sql("""
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_28996\404720421.py:47: UserWarning: pandas only supports SQLAlchemy connec

⚠️ Date filtering error: Invalid comparison between dtype=datetime64[ns] and Timestamp
Events: 2 | Logs: 528
📝 Building KB chunks...
📋 Building LOG summary chunks...
KB chunks: 2 | LOG chunks: 31
Encoding 2 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Index built: 2 chunks, 384 dimensions
✅ Saved kb/faiss_kb.index: 2 chunks
Encoding 31 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Index built: 31 chunks, 384 dimensions
✅ Saved kb/faiss_log.index: 31 chunks
✅ Build complete!


In [17]:
# (moved) log debugging is now in chunks_kb.jsonl / chunks_log.jsonl


In [18]:
# (moved) print counts via the load cells below


In [19]:
#acess role

In [5]:
import json

kb_chunks = []
with open("kb/chunks_kb.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        kb_chunks.append(json.loads(line))

log_chunks = []
with open("kb/chunks_log.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        log_chunks.append(json.loads(line))

print("len(kb_chunks)  =", len(kb_chunks))
print("len(log_chunks) =", len(log_chunks))


FileNotFoundError: [Errno 2] No such file or directory: 'kb/chunks_kb.jsonl'

In [11]:
LOG_HINTS = ["log", "error", "exception", "stack", "trace", "500", "statuscode", "crash", "lỗi", "bug"]

def looks_like_log_question(q: str) -> bool:
    q = (q or "").lower()
    return any(k in q for k in LOG_HINTS)

def _retrieve_from(index_obj, chunk_list, question: str, top_k: int = 5):
    """Retrieve chunks with improved ranking."""
    q = model.encode([f"query: {question}"], normalize_embeddings=True)
    q = np.array(q, dtype="float32")

    # Search wider untuk paraphrase matching (top_k*30 -> top_k*50)
    k = min(index_obj.ntotal, max(top_k * 50, 50))
    scores, ids = index_obj.search(q, k)

    results = []
    seen_events = set()  # Deduplicate by event to avoid repetition
    
    for score, idx in zip(scores[0], ids[0]):
        if idx < 0 or idx >= len(chunk_list):
            continue
        item = chunk_list[idx]
        
        # Avoid duplication from same event (different chunk types)
        event_id = item.get("meta", {}).get("event_id")
        if event_id and event_id in seen_events and len(results) >= top_k:
            continue
        if event_id:
            seen_events.add(event_id)
        
        results.append({
            "score": float(score),
            "text": item["text"],
            "meta": item["meta"]
        })
        if len(results) >= top_k:
            break
    return results

def _is_too_generic(q: str) -> bool:
    q = (q or "").strip().lower()
    return q in ["có sự kiện nào không", "có sự kiện không", "sự kiện", "event", "có gì không"]

def sql_upcoming_published_events(limit: int = 5):
    # fallback for very generic user queries (bypass embeddings)
    try:
        df = pd.read_sql(f"""
            SELECT TOP ({limit})
                e.Title, e.StartTime, e.EndTime, e.Description, e.MaxCapacity, e.Mode
            FROM Event e
            WHERE (e.DeletedAt IS NULL OR e.DeletedAt = '') 
              AND e.Status IN ('Published')
            ORDER BY e.StartTime
        """, get_conn())
        return df
    except Exception as e:
        print(f"⚠️ SQL fallback error: {e}")
        return pd.DataFrame()

def retrieve(question: str, top_k: int = 5, role: str = "user"):
    role = (role or "user").lower().strip()

    # USER: always search KB (events/docs)
    if role == "user":
        return _retrieve_from(kb_index, kb_chunks, question, top_k)

    # ADMIN: route to logs when question looks like troubleshooting/logs
    if looks_like_log_question(question):
        log_results = _retrieve_from(log_index, log_chunks, question, top_k)
        # If minimal results from logs, also search KB
        if len(log_results) < top_k // 2:
            kb_results = _retrieve_from(kb_index, kb_chunks, question, top_k - len(log_results))
            return log_results + kb_results
        return log_results

    # otherwise admin searching knowledge base
    return _retrieve_from(kb_index, kb_chunks, question, top_k)

def retrieve_with_fallback(question: str, top_k: int = 5, role: str = "user"):
    role = (role or "user").lower().strip()
    hits = retrieve(question, top_k=top_k, role=role)

    # If user asked generic "any events?" and retrieval is weak, return structured list
    if role == "user" and (_is_too_generic(question) or len(hits) == 0):
        df = sql_upcoming_published_events(limit=5)
        if len(df) > 0:
            # convert to pseudo hits so your LLM can cite them in context
            pseudo = []
            for i, r in df.iterrows():
                txt = f"""SỰ KIỆN: {r.get('Title','')}

Thời gian: {r.get('StartTime','')} - {r.get('EndTime','')}
Hình thức: {r.get('Mode', 'Offline')}
Sức chứa: {r.get('MaxCapacity', 'TBD')} người

Mô tả: {r.get('Description','')}
"""
                pseudo.append({
                    "score": 1.0,
                    "text": txt,
                    "meta": {"type":"event_sql", "access":"user", "source":"sql_fallback", "row": int(i), "source_kind": "event_basic"}
                })
            return pseudo

    return hits


In [53]:
import os
from dotenv import load_dotenv

# Reload .env file to get fresh API key
load_dotenv(override=True)

GROQ_API_KEY = os.getenv("GROQ_API_KEY", "").strip()

if GROQ_API_KEY:
    print(f"✅ GROQ_API_KEY Loaded: {GROQ_API_KEY[:10]}...{GROQ_API_KEY[-10:]}")
    print(f"✅ Length: {len(GROQ_API_KEY)} characters")
else:
    print("❌ GROQ_API_KEY NOT SET!")


✅ GROQ_API_KEY Loaded: gsk_x7cZCg...AYtTHmQIp6
✅ Length: 56 characters


In [23]:
# ===== VERIFY API KEY =====
import requests

def test_groq_api():
    """Quick test to verify Groq API key works."""
    key = os.getenv("GROQ_API_KEY", "").strip()
    if not key:
        print("❌ No API key found")
        return False
    
    headers = {
        "Authorization": f"Bearer {key}",
        "Content-Type": "application/json",
    }
    
    payload = {
        "model": "llama-3.1-8b-instant",
        "messages": [{"role": "user", "content": "Say 'OK' briefly"}],
        "temperature": 0.2,
        "max_tokens": 10,
    }
    
    try:
        response = requests.post(
            "https://api.groq.com/openai/v1/chat/completions",
            headers=headers,
            json=payload,
            timeout=10
        )
        
        if response.status_code == 200:
            print("✅ Groq API is working!")
            return True
        else:
            print(f"❌ API error {response.status_code}: {response.text}")
            return False
    except Exception as e:
        print(f"❌ Connection error: {e}")
        return False

print("Testing Groq API...")
test_groq_api()


Testing Groq API...
✅ Groq API is working!


True

In [12]:
SYSTEM_PROMPT_USER = (
    "Bạn là CHUYÊN VIÊN TƯ VẤN SỰ KIỆN tại hệ thống AEMS.\n"
    "Bạn có kiến thức sâu về các sự kiện, lịch trình, địa điểm, tài liệu, ĐÁNH GIÁ từ người tham gia.\n"
    "HƯỚNG DẪN:\n"
    "1. CHỈ trả lời dựa vào CONTEXT được cung cấp, KHÔNG sáng tạo thông tin ngoài dữ liệu.\n"
    "2. TRẢ LỜI CHI TIẾT và CÓ GIẢI THÍCH đầy đủ:\n"
    "   - Nêu rõ các thông tin cụ thể (thời gian, địa điểm, chi phí, v.v.)\n"
    "   - Giải thích lý do hoặc bối cảnh nếu có\n"
    "   - KHI RECOMMEND: ƯU TIÊN các sự kiện có RATING CAO (4★+) và nhiều reviews\n"
    "   - Cung cấp các gợi ý hữu ích nếu phù hợp\n"
    "3. TRÍCH DẪN NGUỒN của thông tin (dùng [1], [2]... từ CONTEXT).\n"
    "4. Khi câu hỏi về đánh giá/review: luôn đề cập RATING TRUNG BÌNH, số lượng đánh giá, và những nhận xét tích cực.\n"
    "5. Nếu CONTEXT không chứa thông tin cần, hãy trả lời chính xác: 'Không tìm thấy thông tin này trong dữ liệu hệ thống.'\n"
    "6. Nếu câu hỏi có liên quan nhưng CONTEXT không đủ, hãy nêu những gì bạn tìm được và gợi ý thêm thông tin.\n"
    "7. Luôn TỔN TRỌNG và CHUYÊN NGHIỆP trong lời nói.\n"
    "8. Trả lời bằng TIẾNG VIỆT, rõ ràng, dễ hiểu.\n"
    "9. TRÁNH: câu trả lời ngắn gọn quá, generic hoặc mơ hồ.\n"
)

SYSTEM_PROMPT_ADMIN = (
    "Bạn là CHUYÊN VIÊN HỖ TRỢ ADMIN / QUẢN TRỊ hệ thống AEMS.\n"
    "Bạn có quyền truy cập thông tin sự kiện, nhật ký hệ thống, lỗi, dữ liệu quản lý, và ĐÁNH GIÁ từ người dùng.\n"
    "HƯỚNG DẪN:\n"
    "1. CHỈ trả lời dựa vào CONTEXT, KHÔNG lập dự báo hay sáng tạo thông tin.\n"
    "2. Khi phân tích LỖI/EXCEPTION: cung cấp thông tin chi tiết về nguyên nhân, mức độ, tần suất.\n"
    "3. Khi câu hỏi liên quan đến SỰ KIỆN: cung cấp thông tin quản lý (logistics, tài chính, nhân sự, FEEDBACK).\n"
    "4. TRẢ LỜI CHI TIẾT:\n"
    "   - Nêu rõ con số, thống kê, xu hướng\n"
    "   - Đưa ra khuyến nghị hoặc hành động nếu phù hợp\n"
    "   - Liên kết các vấn đề với nhau (ví dụ: lỗi A gây ra vấn đề B)\n"
    "   - Phân tích RATING/FEEDBACK để đánh giá chất lượng sự kiện\n"
    "5. TRÍCH DẪN nguồn dữ liệu ([1], [2]...) để tăng độ tin cậy.\n"
    "6. Nếu thông tin không có trong CONTEXT, hãy nói rõ và gợi ý bước tiếp theo.\n"
    "7. Luôn CHUYÊN NGHIỆP, ĐẶC THỊ và CÓ HƯỚNG GIẢI QUYẾT.\n"
    "8. Trả lời bằng TIẾNG VIỆT.\n"
)


In [16]:
def _format_context(hits: list, max_chars_each: int = 1500) -> str:
    """
    Format context với metadata đầy đủ để model reference chính xác.
    Tăng max_chars_each từ 900 -> 1500 để preserve chi tiết hơn.
    """
    blocks = []
    for i, h in enumerate(hits, start=1):
        meta = h.get("meta", {}) or {}
        t = (h.get("text", "") or "").strip()
        if not t:
            continue

        # Giữ text dài hơn để context đầy đủ
        if len(t) > max_chars_each:
            t = t[:max_chars_each] + "\n[... xem chi tiết đầy đủ ...]"

        # Trích dẫn metadata
        source_kind = meta.get("source_kind", "")
        ref_info = ""
        
        if source_kind.startswith("event"):
            ref_info = f"sự kiện #{meta.get('event_title', 'N/A')}"
        elif source_kind.startswith("log"):
            ref_info = f"lỗi hệ thống (loại: {meta.get('exception_type','')}, mã: {meta.get('status_code','')})"
        
        block_header = f"[{i}] {meta.get('type', 'info').replace('event_','').replace('_',' ').upper()}"
        if ref_info:
            block_header += f" - {ref_info}"
        
        blocks.append(f"{block_header}\n{t}")
    return "\n\n" + "="*80 + "\n" + "\n\n".join(blocks) + "\n" + "="*80


In [6]:
def call_llm_groq(question: str, hits: list, role: str = "user", model_name="llama-3.1-8b-instant"):
    role = (role or "user").lower().strip()
    key = os.getenv("GROQ_API_KEY", "").strip()
    if not key:
        raise RuntimeError("GROQ_API_KEY is empty.")

    # ✅ Lọc hits theo quyền truy cập bằng can_access(meta, role)
    filtered = []
    for h in (hits or []):
        meta = h.get("meta", {}) or {}
        if can_access(meta, role):
            filtered.append(h)

    # ✅ Nếu sau lọc không còn gì -> trả câu chuẩn (không gọi LLM cho đỡ tốn)
    if len(filtered) == 0:
        yield "Không tìm thấy thông tin trong dữ liệu."
        return

    context = _format_context(filtered)

    system_prompt = SYSTEM_PROMPT_ADMIN if role == "admin" else SYSTEM_PROMPT_USER

    messages = [
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": (
                f"CONTEXT THÔNG TIN HỆ THỐNG:\n{context}\n\n"
                f"CÂU HỎI CỦA BẠN:\n{question}\n\n"
                "YÊU CẦU TRẢ LỜI:\n"
                "- Trả lời CHI TIẾT, đầy đủ, có giải thích.\n"
                "- Chỉ dùng thông tin từ CONTEXT, KHÔNG sáng tạo ngoài dữ liệu.\n"
                "- LUÔN TRÍCH DẪN từ [1], [2]... từ CONTEXT để chứng minh câu trả lời.\n"
                "- Nếu CONTEXT không đủ, hãy nói rõ thiếu thông tin gì.\n"
                "- Giữ tông chuyên nghiệp, hữu ích, và TIẾNG VIỆT chuẩn.\n"
            )
        }
    ]

    headers = {
        "Authorization": f"Bearer {key}",
        "Content-Type": "application/json",
        "Accept": "text/event-stream",
    }

    payload = {
        "model": model_name,
        "messages": messages,
        "temperature": 0.3,  # Thấp hơn một chút (0.2->0.3) để balanced output
        "max_tokens": 2048,  # Tăng token limit để trả lời dài hơn
        "stream": True,
    }

    with requests.post(GROQ_URL, headers=headers, json=payload, stream=True, timeout=60) as r:
        r.encoding = "utf-8"
        if r.status_code != 200:
            raise RuntimeError(f"Groq error {r.status_code}: {r.text}")

        for line in r.iter_lines(decode_unicode=True):
            if not line:
                continue
            if line.startswith("data:"):
                data = line[len("data:"):].strip()
                if data == "[DONE]":
                    break
                try:
                    obj = json.loads(data)
                    delta = obj["choices"][0]["delta"].get("content")
                    if delta:
                        yield delta
                except Exception:
                    continue


In [27]:
app=FastAPI()

In [28]:
from pydantic import BaseModel


In [15]:
class AskReq(BaseModel):
    question: str
    top_k: int = 5
    role: str = ""


In [30]:
@app.get("/health")
def health():
    return {"status": "ok"}

In [ ]:
# ===== SIMPLIFIED AUTO-UPDATE WITHOUT COMPLEX DEPENDENCIES =====
# This version works standalone for periodic updates

from fastapi import FastAPI
from fastapi.responses import StreamingResponse
import nest_asyncio, uvicorn, threading
from datetime import datetime
from apscheduler.schedulers.background import BackgroundScheduler
from apscheduler.triggers.interval import IntervalTrigger
import json

print("🔧 Setting up automatic periodic updates...")

# ===== GLOBAL STATE FOR UPDATES =====
class RAGSystemState:
    def __init__(self):
        self.last_updated = None
        self.last_checked = None
        self.is_updating = False
        self.update_count = 0
        self.status_msg = "Initializing..."
    
    def mark_updated(self):
        self.last_updated = datetime.now()
        self.update_count += 1
    
    def mark_checked(self):
        self.last_checked = datetime.now()
    
    def get_status(self):
        return {
            "is_updating": self.is_updating,
            "last_updated": self.last_updated.isoformat() if self.last_updated else None,
            "last_checked": self.last_checked.isoformat() if self.last_checked else None,
            "update_count": self.update_count,
            "status": self.status_msg,
            "auto_update_interval": "Every 1 hour",
        }

rag_state = RAGSystemState()

# ===== BACKGROUND UPDATE FUNCTION (Simplified) =====
def periodic_sync_data():
    """Simple function: Load data từ DB, update chunks nếu có changes."""
    if rag_state.is_updating:
        print("⚠️ Update already in progress, skipping...")
        return
    
    print(f"\n🔄 [{datetime.now()}] Checking for updates from database...")
    rag_state.is_updating = True
    rag_state.status_msg = "Syncing data..."
    
    try:
        # Reload chunks từ JSONL files (hoặc từ DB nếu muốn rebuild)
        # Đơn giản: Load lại từ files
        global kb_chunks, kb_index, log_chunks, log_index
        
        with open("kb/chunks_kb.jsonl", "r", encoding="utf-8") as f:
            kb_chunks = [json.loads(line) for line in f]
        
        with open("kb/chunks_log.jsonl", "r", encoding="utf-8") as f:
            log_chunks = [json.loads(line) for line in f]
        
        # Nếu muốn rebuild indexes: reload từ disk
        # kb_index = faiss.read_index("kb/faiss_kb.index")
        # log_index = faiss.read_index("kb/faiss_log.index")
        
        rag_state.mark_updated()
        rag_state.status_msg = f"✅ Synced {len(kb_chunks)} KB chunks + {len(log_chunks)} LOG chunks"
        print(f"✅ Periodic sync complete! Updated chunks: {len(kb_chunks)} KB, {len(log_chunks)} LOG")
        
    except Exception as e:
        rag_state.status_msg = f"❌ Sync error: {str(e)}"
        print(f"❌ Sync error: {e}")
    finally:
        rag_state.is_updating = False
        rag_state.mark_checked()

# ===== FASTAPI APP WITH AUTO-UPDATE =====
app = FastAPI(title="RAG API - Auto-Update Mode")

@app.post("/ask")
def ask(req: AskReq):
    """Query RAG with detailed consultant response."""
    top_k = req.top_k if req.top_k > 0 else 10
    hits = retrieve_with_fallback(req.question, top_k=top_k, role=req.role)
    
    answer_parts = []
    try:
        for tok in call_llm_groq(req.question, hits, role=req.role):
            answer_parts.append(tok)
    except Exception as e:
        return {
            "question": req.question,
            "answer": "",
            "error": str(e),
            "last_updated": rag_state.last_updated.isoformat() if rag_state.last_updated else None
        }
    
    return {
        "question": req.question,
        "answer": "".join(answer_parts),
        "sources": [{"score": h.get("score"), "meta": h.get("meta")} for h in hits],
        "total_hits": len(hits),
        "last_updated": rag_state.last_updated.isoformat() if rag_state.last_updated else None
    }

@app.post("/ask_stream")
def ask_stream(req: AskReq):
    """Streaming response mode."""
    top_k = req.top_k if req.top_k > 0 else 10
    hits = retrieve_with_fallback(req.question, top_k=top_k, role=req.role)
    
    def gen():
        try:
            for tok in call_llm_groq(req.question, hits, role=req.role):
                yield tok
        except Exception as e:
            yield f"\n[LỖI] {str(e)}"
    
    return StreamingResponse(gen(), media_type="text/plain")

@app.post("/rebuild")
def manual_rebuild():
    """Manually trigger data sync/rebuild."""
    if rag_state.is_updating:
        return {
            "status": "updating",
            "message": "Update already in progress",
            "last_updated": rag_state.last_updated.isoformat() if rag_state.last_updated else None
        }
    
    # Run rebuild in background thread
    def sync_thread():
        periodic_sync_data()
    
    threading.Thread(target=sync_thread, daemon=True).start()
    
    return {
        "status": "building",
        "message": "Data sync started in background",
        "previous_update": rag_state.last_updated.isoformat() if rag_state.last_updated else None
    }

@app.get("/status")
def get_status():
    """Get system status and auto-update info."""
    return {
        "system": "RAG Event Consultant (Auto-Update)",
        "is_updating": rag_state.is_updating,
        "last_updated": rag_state.last_updated.isoformat() if rag_state.last_updated else "Never",
        "last_checked": rag_state.last_checked.isoformat() if rag_state.last_checked else "Never",
        "total_updates": rag_state.update_count,
        "status": rag_state.status_msg,
        "auto_update_interval": "1 hour (configurable)",
        "api_endpoints": {
            "ask": "POST /ask - Get RAG response",
            "ask_stream": "POST /ask_stream - Stream response",
            "rebuild": "POST /rebuild - Manual trigger",
            "status": "GET /status - System status",
            "health": "GET /health - Health check"
        }
    }

@app.get("/health")
def health():
    return {
        "status": "ok",
        "model": "RAG Event Consultant (Auto-Update Ready)",
        "is_updating": rag_state.is_updating,
        "last_update": rag_state.last_updated.isoformat() if rag_state.last_updated else None
    }

# ===== STARTUP: Try to load existing indexes =====
try:
    with open("kb/chunks_kb.jsonl", "r", encoding="utf-8") as f:
        kb_chunks = [json.loads(line) for line in f]
    kb_index = faiss.read_index("kb/faiss_kb.index")
    print(f"✅ Loaded {len(kb_chunks)} KB chunks from disk")
except:
    print("⚠️ No existing indexes found - will be created on first rebuild")
    kb_chunks = []
    kb_index = None

try:
    with open("kb/chunks_log.jsonl", "r", encoding="utf-8") as f:
        log_chunks = [json.loads(line) for line in f]
    log_index = faiss.read_index("kb/faiss_log.index")
    print(f"✅ Loaded {len(log_chunks)} LOG chunks from disk")
except:
    log_chunks = []
    log_index = None

# ===== SETUP SCHEDULER FOR PERIODIC UPDATES =====
nest_asyncio.apply()

scheduler = BackgroundScheduler()
scheduler.add_job(
    periodic_sync_data,
    IntervalTrigger(hours=1),  # Update mỗi 1 giờ (thay đổi: minutes=5, hours=6, etc.)
    id='rag_periodic_sync',
    name='RAG Periodic Data Sync',
    replace_existing=True,
    coalesce=True,  # Prevent multiple jobs if scheduler falls behind
    max_instances=1  # Only one sync at a time
)
scheduler.start()
print(f"✅ Background scheduler started - data will sync every 1 hour")

# ===== START FASTAPI SERVER =====
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

threading.Thread(target=run_server, daemon=True).start()
print("✅ RAG API Server running at http://127.0.0.1:8000")
print("📊 Auto-Update System ACTIVE - Syncs every 1 hour")
print("🔗 Check /status or /health endpoints for info")


🔧 Setting up automatic periodic updates...
⚠️ No existing indexes found - will be created on first rebuild
✅ Background scheduler started - data will sync every 1 hour
✅ RAG API Server running at http://127.0.0.1:8000
📊 Auto-Update System ACTIVE - Syncs every 1 hour
🔗 Check /status or /health endpoints for info


INFO:     Started server process [29260]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


In [32]:
import os
print("GROQ_API_KEY exists:", bool(os.getenv("GROQ_API_KEY")))


GROQ_API_KEY exists: True


INFO:     Started server process [28996]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


In [33]:
#demo

In [34]:
import requests
with requests.post("http://127.0.0.1:8000/ask_stream", json={"question":"có những lỗi gì trong hệ thống", "top_k":3, "role":"admin"}, stream=True) as r:
    for chunk in r.iter_content(chunk_size=None):
        if chunk:
            print(chunk.decode("utf-8"), end="")


INFO:     127.0.0.1:52434 - "POST /ask_stream HTTP/1.1" 200 OK
Có những lỗi sau trong hệ thống:

- InvalidOperationException khi không tìm thấy hồ sơ sinh viên (mục [1]).
- Exception khi Email hoặc mật khẩu không chính xác hoặc tài khoản bị khóa (mục [2]).
- Exception khi Email hoặc mật khẩu không chính xác (mục [3]).

In [35]:
# Demo: show a few KB chunks and LOG summary chunks
print("=== KB SAMPLE ===")
for i, item in enumerate(kb_chunks[:3]):
    m = item["meta"]
    print("="*80)
    print("CHUNK", i, "| type:", m.get("type"), "| access:", m.get("access"))
    print("event_id:", m.get("event_id"))
    print(item["text"][:500])

print("\n=== LOG SUMMARY SAMPLE ===")
for i, item in enumerate(log_chunks[:3]):
    m = item["meta"]
    print("="*80)
    print("CHUNK", i, "| type:", m.get("type"), "| access:", m.get("access"))
    print("exception_type:", m.get("exception_type"), "| status_code:", m.get("status_code"), "| count:", m.get("count"))
    print(item["text"][:500])


=== KB SAMPLE ===
CHUNK 0 | type: event | access: user
event_id: 0D585A2C-40AA-4C0C-B4E9-B097AF559C4F
EVENT OVERVIEW
Title: AI Workshop 2026
Description: Workshop giới thiệu về AI và Machine Learning.
Time: 2026-03-10 08:00:00 - 2026-03-10 11:00:00
Status: Published

LOCATION DETAILS
- Room A101
  Address: Building A - Floor 1
  Capacity: 150
  Type: 1
- Room B202
  Address: Building B - Floor 2
  Capacity: 100
  Type: 1
CHUNK 1 | type: event | access: user
event_id: 24A67A53-5126-4793-864F-36A33DF5AC40
EVENT OVERVIEW
Title: Software Testing
Description: Khóa học chuyên sâu
Time: 2026-04-05 13:00:00 - 2026-04-05 17:00:00
Status: Published

LOCATION DETAILS
- Room A101
  Address: Building A - Floor 1
  Capacity: 150
  Type: 1
- Room B202
  Address: Building B - Floor 2
  Capacity: 100
  Type: 1

=== LOG SUMMARY SAMPLE ===
CHUNK 0 | type: system_log_summary | access: admin
exception_type: ArgumentException | status_code: 500.0 | count: 346
SYSTEM ERROR SUMMARY
StatusCode: 500.0
Exception

In [36]:
print("kb_index.ntotal  =", kb_index.ntotal)
print("log_index.ntotal =", log_index.ntotal)
print("len(kb_chunks)   =", len(kb_chunks))
print("len(log_chunks)  =", len(log_chunks))


kb_index.ntotal  = 2
log_index.ntotal = 31
len(kb_chunks)   = 2
len(log_chunks)  = 31


In [36]:
# ===== COMPREHENSIVE TEST WITH IMPROVED RAG =====
import logging
logging.basicConfig(level=logging.INFO)

test_questions = [
    ("tôi muốn biết địa điểm tổ chức 1 số sự kiện", "user"),
   
]

print("="*80)
print("🚀 RAG SYSTEM TEST - EVENT CONSULTANT MODE")
print("="*80)

for question, role in test_questions:
    print("\n" + "="*80)
    print(f"👤 Role: {role.upper()}")
    print(f"❓ Question: {question}")
    print("="*80)
    
    # Retrieve with improved top_k
    hits = retrieve_with_fallback(question, top_k=10, role=role)
    print(f"✅ Retrieved {len(hits)} relevant chunks\n")
    
    try:
        print("📋 ANSWER:")
        print("-"*80)
        output = ""
        for tok in call_llm_groq(question, hits, role=role):
            print(tok, end="", flush=True)
            output += tok
        print("\n")
    except Exception as e:
        print(f"❌ Error: {type(e).__name__}: {e}\n")

print("="*80)
print("✅ Test completed!")
print("="*80)


🚀 RAG SYSTEM TEST - EVENT CONSULTANT MODE

👤 Role: USER
❓ Question: tôi muốn biết địa điểm tổ chức 1 số sự kiện


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Retrieved 4 relevant chunks

📋 ANSWER:
--------------------------------------------------------------------------------
Tôi có thể giúp bạn tìm thông tin về địa điểm tổ chức một số sự kiện.

Từ CONTEXT, tôi thấy có 2 sự kiện được cung cấp thông tin về địa điểm tổ chức:

1. Sự kiện #AI Workshop 2026:
 - Địa điểm tổ chức: Room A101
 - Địa chỉ: Building A - Floor 1
 - Sức chứa: 150 người
 - Loại: 1
 - Trạng thái: Maintenance
 - Mô tả: Phòng A101

[1] LOCATION - sự kiện #AI Workshop 2026

2. Sự kiện #Software Testing:
 - Địa điểm tổ chức: Room B202
 - Địa chỉ: Building B - Floor 2
 - Sức chứa: 100 người
 - Loại: 1
 - Trạng thái: Maintenance
 - Mô tả: Phòng B202 dùng cho bootcamp.

[2] LOCATION - sự kiện #Software Testing

Vì CONTEXT chỉ cung cấp thông tin về 2 sự kiện này, nên tôi không thể cung cấp thông tin về địa điểm tổ chức các sự kiện khác.

Nếu bạn muốn biết thông tin về các sự kiện khác, vui lòng cung cấp thêm thông tin về các sự kiện đó.

✅ Test completed!


In [65]:
# ===== DEMO: Enhanced RAG System with Feedback Rating =====
print("="*80)
print("🎯 DEMO: RAG System with Feedback Rating & Recommendation")
print("="*80)

# First, show current state
print("\n📊 CURRENT STATE:")
print(f"✅ Event Overview chunks: {len([c for c in kb_chunks if c.get('meta', {}).get('type') == 'event_overview'])}")
print(f"✅ Feedback chunks: {len([c for c in kb_chunks if c.get('meta', {}).get('type') == 'event_feedback'])}")
print(f"✅ Total KB chunks: {len(kb_chunks)}")

# Show sample event overview with feedback metadata
print("\n📝 Sample Event Overview (with feedback metadata):")
for i, item in enumerate(kb_chunks[:1]):
    m = item.get("meta", {})
    print(f"\nChunk {i+1}: {m.get('type')}")
    print(f"Event: {m.get('event_title')}")
    print(f"Rating: {m.get('avg_rating', 0)}/5 ({m.get('review_count', 0)} reviews)")
    print(f"Text preview:\n{item.get('text', '')[:400]}...\n")

# Demo: How it works when feedback exists
print("\n💡 HOW IT WORKS WITH FEEDBACK DATA:")
print("-"*80)
print("""
When users ask questions like:
1. "Sự kiện nào được review tốt nhất?" 
2. "Sự kiện nào có đánh giá cao?"
3. "Sự kiện nào được mọi người yêu thích?"

The system will:
✅ Retrieve events with FEEDBACK chunks
✅ Prioritize events with avg_rating >= 4.0 stars
✅ Include star ratings and review counts in recommendations
✅ Cite specific feedback examples from users
✅ Explain why an event is highly rated
""")

# Show how metadata is used for ranking
print("\n🔍 FEEDBACK METADATA IN CHUNKS:")
print("-"*80)
for i, item in enumerate(kb_chunks):
    m = item.get("meta", {})
    if m.get('avg_rating', 0) > 0:
        print(f"Event: {m.get('event_title')}")
        print(f"  ⭐ Rating: {m.get('avg_rating', 0)}/5")
        print(f"  📊 Reviews: {m.get('review_count', 0)} people")

print("\n" + "="*80)
print("✅ Feedback Integration Complete!")
print("="*80)
print("""
NEXT STEPS:
1. Add sample feedback data to your database (Feedback table)
2. Re-run the build process to create feedback chunks
3. Ask questions like:
   - "Sự kiện nào được review cao nhất?"
   - "Sự kiện nào mọi người đánh giá tốt?"
   - "Tôi muốn sự kiện có rating 4+ sao"
   
The RAG system will automatically recommend based on feedback ratings!
""")


🎯 DEMO: RAG System with Feedback Rating & Recommendation

📊 CURRENT STATE:
✅ Event Overview chunks: 2
✅ Feedback chunks: 0
✅ Total KB chunks: 4

📝 Sample Event Overview (with feedback metadata):

Chunk 1: event_overview
Event: AI Workshop 2026
Rating: 0/5 (0 reviews)
Text preview:
SỰ KIỆN: AI Workshop 2026

MÔ TẢ:
Workshop giới thiệu về AI và Machine Learning.

THÔNG TIN CƠ BẢN:
- Bắt đầu: 2026-03-10 08:00:00
- Kết thúc: 2026-03-10 11:00:00
- Hình thức: None
- Sức chứa: 120 người
- Loại sự kiện: Workshop
- Phòng ban: Information Technology
- Chủ đề: 
- Người tổ chức: N/A...


💡 HOW IT WORKS WITH FEEDBACK DATA:
--------------------------------------------------------------------------------

When users ask questions like:
1. "Sự kiện nào được review tốt nhất?" 
2. "Sự kiện nào có đánh giá cao?"
3. "Sự kiện nào được mọi người yêu thích?"

The system will:
✅ Retrieve events with FEEDBACK chunks
✅ Prioritize events with avg_rating >= 4.0 stars
✅ Include star ratings and review counts in re